In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
sc = spark.sparkContext

In [3]:
orders_base_rdd = sc.textFile("data/retail_db_orders_part-00000")
customers_base_rdd = sc.textFile("data/retail_db_customers_part-00000")
order_items_base_rdd = sc.textFile("data/retail_db_order_items_part-00000")

## 5. No. of customers who have spent more than Dollar 1000 in total

In [4]:
# order_id, order_customer_id
order_cols = orders_base_rdd.map(lambda x: (x.split(",")[0], x.split(",")[2]))

In [5]:
# order_id, order_item _subtotal
order_items_cols = order_items_base_rdd.map(lambda x: (x.split(",")[1], float(x.split(",")[4])))

In [6]:
order_items_cols.take(5)

[('1', 299.98), ('2', 199.99), ('2', 250.0), ('2', 129.99), ('4', 49.98)]

In [10]:
order_items_col_new = order_cols.join(order_items_cols)

In [11]:
order_items_col_new.take(10)

[('34566', ('3066', 250.0)),
 ('34566', ('3066', 179.97)),
 ('34577', ('7733', 299.98)),
 ('34577', ('7733', 299.95)),
 ('34577', ('7733', 299.98)),
 ('34577', ('7733', 59.99)),
 ('34577', ('7733', 79.98)),
 ('34583', ('1558', 44.99)),
 ('34583', ('1558', 239.96)),
 ('34583', ('1558', 129.99))]

In [12]:
order_items_col_cal = order_items_col_new.map(lambda x: x[1]).reduceByKey(lambda x,y: x+y)

In [13]:
order_items_col_cal.take(10)

[('10436', 2484.63),
 ('9419', 4219.44),
 ('833', 2498.7),
 ('8506', 4443.4800000000005),
 ('10173', 1077.78),
 ('3789', 2659.6500000000005),
 ('10709', 839.9000000000001),
 ('1285', 6184.370000000001),
 ('26', 3209.8),
 ('5023', 2289.7200000000003)]

In [14]:
order_items_col_cal.filter(lambda x: x[1]>1000).count()

11148

In [15]:
spark.stop()